**Scraping des données du site Rekrute à Marrakech**

In [62]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

Rekrute base URL

In [63]:
#url = "https://www.rekrute.com/en/offres.html?contractType=7&keyword=c&o=1&p=1&query=c&s=1&studyLevelId=bac2&workExperienceId%5B0%5D=3"


In [64]:
url = "https://www.rekrute.com/offres.html?st=d&keywordNew=1&jobLocation=RK&tagSearchKey=&keyword=marrakech"


In [65]:
params = {
    "keyword": "Marrakech",
    "query": "Marrakech"
}

In [66]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

print("Code HTTP :", response.status_code)
print("URL :", response.url)

soup = BeautifulSoup(response.text, "html.parser")

Code HTTP : 200
URL : https://www.rekrute.com/offres.html?st=d&keywordNew=1&jobLocation=RK&tagSearchKey=&keyword=marrakech&keyword=Marrakech&query=Marrakech


In [67]:
resultats = []

for page in range(1, 6):

    page_url = url + f"&p={page}"

    response = requests.get(
        page_url,
        headers=headers,
        timeout=20
    )

    soup_page = BeautifulSoup(
        response.text,
        "html.parser"
    )

    offres = soup_page.find_all(
        "li",
        class_="post-id"
    )

    for bloc in offres:

        lien_elem = bloc.find(
            "a",
            class_="titreJob"
        )

        if not lien_elem:
            continue

        titre = lien_elem.get_text(
            " ",
            strip=True
        )

        href = lien_elem.get(
            "href",
            ""
        )

        offre_url = urljoin(
            page_url,
            href
        )

        localisation = "Inconnue"

        match = re.search(
            r"\|\s*([^(|]+?)\s*\(",
            titre
        )

        if match:
            localisation = match.group(1).strip()

        resultats.append({
            "poste": titre,
            "localisation": localisation,
            "url": offre_url
        })


df = pd.DataFrame(resultats)

df = df.drop_duplicates(
    subset="url"
).reset_index(drop=True)

df.head(20)

,poste,localisation,url
0,Technicien(ne) en Maroquinerie – Contrôle Qual...,Marrakech,https://www.rekrute.com/offre-emploi-technicie...
1,Guides / Médiateurs Culturels | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-guides-me...
2,Graphiste | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-graphiste...
3,Caissier (ère) | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-caissier-...
4,Responsable Commercial Immobilier (H/F) | Marr...,Marrakech,https://www.rekrute.com/offre-emploi-responsab...
5,Responsable prospection - Marrakech | Marrakec...,Marrakech,https://www.rekrute.com/offre-emploi-responsab...
6,Commercial Terrain | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-commercia...
7,Directeur(trice) Marketing et Communication | ...,Marrakech,https://www.rekrute.com/offre-emploi-directeur...
8,Chef de projet Communication Junior (Marketing...,Marrakech,https://www.rekrute.com/offre-emploi-chef-de-p...
9,Chargé(e) des Opérations RH | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-chargee-d...


In [68]:
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))

Nombre de lignes : 50
Nombre de colonnes : 3


#  Nettoyage des données

1. Vérifier les doublons

In [69]:
print("Doublons détectés (sur 'url') :", df.duplicated(subset="url").sum())


Doublons détectés (sur 'url') : 0


2. Supprimer les doublons

In [70]:
df = df.drop_duplicates(subset="url").copy()
print("Nombre d'offres après suppression des doublons :", len(df))


Nombre d'offres après suppression des doublons : 50


3. Vérifier les valeurs manquantes

In [71]:
df.isnull().sum()


poste           0
localisation    0
url             0
dtype: int64

4. Supprimer les lignes incomplètes (poste ou url manquants)

In [72]:
df = df.dropna(subset=["poste", "url"]).copy()
print("Nombre d'offres après suppression des lignes incomplètes :", len(df))


Nombre d'offres après suppression des lignes incomplètes : 50


5. Nettoyer les intitulés de poste (espaces inutiles)

In [73]:
df["poste"] = df["poste"].astype("string").str.strip()
# Normaliser les espaces multiples (ex: double espace) à l'intérieur du texte
df["poste"] = df["poste"].str.replace(r"\s+", " ", regex=True)


6. Nettoyer la colonne localisation

In [74]:
if "localisation" in df.columns:
    df["localisation"] = df["localisation"].astype("string").str.strip()


7. Nettoyer les URLs

In [75]:
# Supprimer les espaces et s'assurer que l'URL est bien une chaîne de caractères
df["url"] = df["url"].astype("string").str.strip()


8. Vérifier que les offres concernent bien Marrakech

In [76]:
# Une offre est jugée pertinente si "Marrakech" apparaît dans le poste OU la localisation
mask_marrakech = (
    df["poste"].str.contains("Marrakech", case=False, na=False)
    | df["localisation"].astype("string").str.contains("Marrakech", case=False, na=False)
)
print("Offres hors Marrakech détectées :", (~mask_marrakech).sum())
df = df[mask_marrakech].copy()


Offres hors Marrakech détectées : 4


9. Vérifier les types de colonnes

In [77]:
df.dtypes


poste           string
localisation    string
url             string
dtype: object

10. Réinitialiser l'index après nettoyage

In [78]:
df = df.reset_index(drop=True)


11. Résumé avant / après nettoyage

In [79]:
print("Valeurs manquantes restantes :")
print(df.isnull().sum())
print()
print("Doublons restants (url) :", df.duplicated(subset="url").sum())
print("Nombre final d'offres après nettoyage :", len(df))
df.head(20)


Valeurs manquantes restantes :
poste           0
localisation    0
url             0
dtype: int64

Doublons restants (url) : 0
Nombre final d'offres après nettoyage : 46


,poste,localisation,url
0,Technicien(ne) en Maroquinerie – Contrôle Qual...,Marrakech,https://www.rekrute.com/offre-emploi-technicie...
1,Guides / Médiateurs Culturels | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-guides-me...
2,Graphiste | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-graphiste...
3,Caissier (ère) | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-caissier-...
4,Responsable Commercial Immobilier (H/F) | Marr...,Marrakech,https://www.rekrute.com/offre-emploi-responsab...
5,Responsable prospection - Marrakech | Marrakec...,Marrakech,https://www.rekrute.com/offre-emploi-responsab...
6,Commercial Terrain | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-commercia...
7,Directeur(trice) Marketing et Communication | ...,Marrakech,https://www.rekrute.com/offre-emploi-directeur...
8,Chef de projet Communication Junior (Marketing...,Marrakech,https://www.rekrute.com/offre-emploi-chef-de-p...
9,Chargé(e) des Opérations RH | Marrakech (Maroc),Marrakech,https://www.rekrute.com/offre-emploi-chargee-d...


In [80]:
df.to_csv(
    "offres_rekrute_marrakech.csv",
    index=False,
    encoding="utf-8-sig"
)

In [81]:
df.to_excel("offres_marrakech.xlsx", index=False)

print("Fichier Excel créé avec succès !")

Fichier Excel créé avec succès !
